In [0]:
client_id = dbutils.secrets.get(scope="ecommerce-project", key="sp-client-id")
tenant_id = dbutils.secrets.get(scope="ecommerce-project", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="ecommerce-project", key="sp-client-secret")

spark.conf.set("fs.azure.account.auth.type.ecommercelakehouse01.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.ecommercelakehouse01.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.ecommercelakehouse01.dfs.core.windows.net", client_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.ecommercelakehouse01.dfs.core.windows.net", client_secret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.ecommercelakehouse01.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
%pip install faker

In [0]:
from faker import Faker
import random
import uuid
from datetime import datetime

fake = Faker()

def generate_order():
    order = {
        "order_id": str(uuid.uuid4()),
        "customer_id": fake.uuid4(),
        "order_timestamp": datetime.now().isoformat(),
        "product_category": random.choice(["Electronics", "Clothing", "Home & Kitchen", "Books", "Sports"]),
        "product_name": fake.word().capitalize() + " " + random.choice(["Pro", "Max", "Lite", "Standard"]),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(5.0, 500.0), 2),
        "payment_method": random.choice(["Credit Card", "Debit Card", "PayPal", "Cash on Delivery"]),
        "customer_country": fake.country()
    }
    return order

In [0]:
orders_batch = [generate_order() for _ in range(100)]
df = spark.createDataFrame(orders_batch)
df.show(5)

In [0]:
df.write.format("parquet").mode("append").save("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/orders/")

In [0]:
dbutils.fs.ls("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/orders/")